In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

csv_path = "phishing_site_urls.csv"
print("cwd:", os.getcwd())
print("exists:", os.path.exists(csv_path))
print("size:", os.path.getsize(csv_path) if os.path.exists(csv_path) else "N/A")

# Show first few bytes to confirm file content
with open(csv_path, "rb") as f:
    print(f.read(512))

# Attempt to read with pandas, but handle empty/invalid
try:
    df = pd.read_csv(csv_path)
    print("Original shape:", df.shape)

    # Remove duplicates
    df = df.drop_duplicates()
    print("After removing duplicates:", df.shape)

    # Split data into training (80%) and testing (20%)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)

    # Optionally, save the splits to new CSV files
    train_df.to_csv("train_data.csv", index=False)
    test_df.to_csv("test_data.csv", index=False)

    display(train_df.head())
except Exception as e:
    print("pandas failed to read CSV:", type(e), e)
    df = None


cwd: c:\Users\ZBOOK\Desktop\Computer Science\Year 3\Sem2\Machine Learning\ScamDetection
exists: True
size: 31567326
b'URL,Label\r\nnobell.it/70ffb52d079109dca5664cce6f317373782/login.SkyPe.com/en/cgi-bin/verification/login/70ffb52d079109dca5664cce6f317373/index.php?cmd=_profile-ach&outdated_page_tmpl=p/gen/failed-to-load&nav=0.5.1&login_access=1322408526,bad\r\nwww.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrcmd=_home-customer&nav=1/loading.php,bad\r\nserviciosbys.com/paypal.cgi.bin.get-into.herf.secure.dispatch35463256rzr321654641dsf654321874/href/href/href/secure/center/update/limit/seccure/4d7a1ff5c55825a2e632a679c2fd5353/,bad\r\n'
Original shape: (549346, 2)
After removing duplicates: (507196, 2)
Train shape: (405756, 2)
Test shape: (101440, 2)


,URL,Label
57714,www.cityoflaredo.com/airport/,good
80839,www.cs.umb.edu/~laur/ARMiner/,good
359375,imdb.com/name/nm1027430/bio,good
428999,seatplans.com/airlines/Air-Transat,good
508916,vehisdidnsu.ru/gate.php,bad


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assign features (X) and target (y)
X = train_df['URL']  # Features: URLs (text data)
y = train_df['Label']  # Target: labels

# Convert text URLs to numerical features using TF-IDF
vectorizer = TfidfVectorizer()
X_vectorized = vectorizer.fit_transform(X)

# Initialize the Random Forest model
rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)

# Define K-Fold cross-validation (e.g., 5 folds)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Perform K-Fold cross-validation and print average accuracy
cv_scores = cross_val_score(rf_model, X_vectorized, y, cv=kfold, scoring='accuracy')
print(f"K-Fold Cross-Validation Accuracy: {cv_scores.mean():.2f} (±{cv_scores.std():.2f})")

# Train the model on the full training set
rf_model.fit(X_vectorized, y)

# Prepare test data
X_test = test_df['URL']
y_test = test_df['Label']
X_test_vectorized = vectorizer.transform(X_test)

# Predict on the test set
y_pred = rf_model.predict(X_test_vectorized)

# Evaluate the model
print("\nTest Set Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


K-Fold Cross-Validation Accuracy: 0.77 (±0.00)

Test Set Accuracy: 0.7761829652996846


c:\Users\ZBOOK\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ZBOOK\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Classification Report:
               precision    recall  f1-score   support

         bad       0.00      0.00      0.00     22704
        good       0.78      1.00      0.87     78736

    accuracy                           0.78    101440
   macro avg       0.39      0.50      0.44    101440
weighted avg       0.60      0.78      0.68    101440



c:\Users\ZBOOK\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
